# Import libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Data Preparation

## Load data

In [2]:
# Define data directory
data_path = Path("/Users/anhtuyet15999/Projects/ecommerce_delivery_time/data/raw")

# Load datasets
customers = pd.read_csv(data_path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(data_path / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(data_path / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(data_path / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(data_path / "olist_orders_dataset.csv")
products = pd.read_csv(data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(data_path / "product_category_name_translation.csv")

## Create dictionary for the dataset

In [3]:
data = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

## Create a copy of dataset

In [4]:
copy = {name: df.copy() for name, df in data.items()}

## Rename Order Date Columns
Use consistent names for timestamps in the working copy. Raw files retain their original column names.

In [5]:
copy["orders"] = copy["orders"].rename(columns={
    "order_purchase_timestamp": "order_purchased_at",
    "order_delivered_carrier_date": "order_delivered_carrier_at",
    "order_delivered_customer_date": "order_delivered_customer_at"
})

## Check overview of data

In [6]:
overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "exact_duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
    }
    for name, df in copy.items()
])

display(overview)

,table,rows,columns,exact_duplicate_rows,missing_cells
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,0,145903
5,orders,99441,8,0,4908
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


## Check missing values

In [7]:
for name, df in copy.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")

    display(pd.DataFrame({
        "missing_count": missing,
        "missing_percent": (
            missing / len(df) * 100
        ).round(2),
    }))


customers


,missing_count,missing_percent



geolocation


,missing_count,missing_percent



order_items


,missing_count,missing_percent


... (8 outputs left out)


sellers


,missing_count,missing_percent



category_translation


,missing_count,missing_percent


## Check table keys

In [8]:
table_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "category_translation": ["product_category_name"],
}

key_checks = []

for name, keys in table_keys.items():
    df = copy[name]

    key_checks.append({
        "table": name,
        "key": " + ".join(keys),
        "rows_with_missing_key": (
            df[keys].isna().any(axis=1).sum()
        ),
        "duplicate_keys": df.duplicated(keys).sum(),
    })

display(pd.DataFrame(key_checks))

,table,key,rows_with_missing_key,duplicate_keys
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,products,product_id,0,0
3,sellers,seller_id,0,0
4,order_items,order_id + order_item_id,0,0
5,order_payments,order_id + payment_sequential,0,0
6,category_translation,product_category_name,0,0


## Convert date to datetime

In [9]:
date_columns = {
    "orders": [
        "order_purchased_at",
        "order_approved_at",
        "order_delivered_carrier_at",
        "order_delivered_customer_at",
        "order_estimated_delivery_date",
    ],
    "order_items": [
        "shipping_limit_date",
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

In [10]:
for name, columns in date_columns.items():
    for col in columns:
        copy[name][col] = pd.to_datetime(
            copy[name][col],
            errors="raise"
        )

In [11]:
str(copy[name][col].dtype)

'datetime64[us]'

## Convert Zipcode to string and change the misspelled column name

In [12]:
# Convert ZIP codes to strings and add leading zeros
copy["customers"]["customer_zip_code_prefix"] = (
    copy["customers"]["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

copy["sellers"]["seller_zip_code_prefix"] = (
    copy["sellers"]["seller_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

copy["geolocation"]["geolocation_zip_code_prefix"] = (
    copy["geolocation"]["geolocation_zip_code_prefix"]
    .astype("string")
    .str.zfill(5)
)

In [13]:
# Correct misspelled column names
copy["products"] = copy["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

In [14]:
copy["products"].columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

## Check missing values in products

In [15]:
# Count missing values in each column
copy["products"].isna().sum()

product_id                      0
product_category_name         610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [16]:
# View products with missing categories
copy["products"][
    copy["products"]["product_category_name"].isna()
].head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [17]:
# View products with zero or negative weights
copy["products"][
    copy["products"]["product_weight_g"] <= 0
]

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


## Check price and freight values

In [18]:
# Check mising values
copy["order_items"][["price", "freight_value"]].isna().sum()

price            0
freight_value    0
dtype: int64

In [19]:
copy["order_items"][["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [20]:
copy["order_items"][
    copy["order_items"]["price"] <= 0
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [21]:
copy["order_items"][
    copy["order_items"]["freight_value"] < 0
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


## Check payment values

In [22]:
copy["order_payments"].isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [23]:
copy["order_payments"]["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [24]:
copy["order_payments"][
    copy["order_payments"]["payment_value"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [25]:
copy["order_payments"][
    copy["order_payments"]["payment_installments"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


## Check order dates

In [26]:
copy["orders"][[
    "order_purchased_at",
    "order_approved_at",
    "order_delivered_carrier_at",
    "order_delivered_customer_at",
    "order_estimated_delivery_date"
]].isna().sum()

order_purchased_at                  0
order_approved_at                 160
order_delivered_carrier_at       1783
order_delivered_customer_at      2965
order_estimated_delivery_date       0
dtype: int64

In [27]:
# Check orders whose deliveried date smaller than purchase date
copy["orders"][
    copy["orders"]["order_delivered_carrier_at"]
    < copy["orders"]["order_purchased_at"]
]

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,e6f5b234bb0d847f10eebd70130c5d49,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,72fcbb1145f2889eddcba6d6c1d6c090,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,ccd8e3459ad58ae538c9d2cf35532ba4,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,5c427f06e7fcf902e309743516e2c580,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
...,...,...,...,...,...,...,...,...
98172,f7780ea2807db31691e83f0013294035,3088cc4bfafe311f1d84830808c2aa6d,delivered,2018-07-30 15:22:15,2018-07-30 15:35:16,2018-07-30 15:00:00,2018-08-02 18:32:30,2018-08-02
98430,d7646ffe8fdd9e7d9557f9f7cbf04530,f896e70d84f6db9a56b86a47dd8718f0,delivered,2018-05-04 14:50:37,2018-05-04 15:10:22,2018-05-04 14:48:00,2018-05-08 19:06:42,2018-05-16
98672,5ded8a3706eabd813685534724f066de,460316e38d4d241fc79d4f6729f8f7f6,delivered,2018-07-18 08:46:52,2018-07-18 09:01:48,2018-07-18 08:44:00,2018-07-25 13:53:17,2018-08-10
98780,d10046876c7d9f01613da59ffa6cb07f,6559cb5b365f173915f766fefcd86972,delivered,2018-07-18 16:14:16,2018-07-18 16:25:17,2018-07-18 15:34:00,2018-07-23 20:46:44,2018-08-07


In [28]:
# Check if orders have delivered_customer_date before delivered_carrier_date
copy["orders"][
    copy["orders"]["order_delivered_customer_at"]
    < copy["orders"]["order_delivered_carrier_at"]
]

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,01c843a2c0600def0b7693dba47af460,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,3947a361301f2ff0f3223159a0f2701c,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,63be4feff10a0b1d85f2cfbf10df9754,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [29]:
# Check if delivered order have no delivered date
copy["orders"][
    (copy["orders"]["order_status"] == "delivered")
    & (copy["orders"]["order_delivered_customer_at"].isna())
]

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


## Check order reviews

In [30]:
copy["order_reviews"].isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [31]:
copy["order_reviews"]["review_score"].value_counts(
    dropna=False
).sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [32]:
copy["order_reviews"][
    (copy["order_reviews"]["review_score"] < 1)
    | (copy["order_reviews"]["review_score"] > 5)
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [33]:
copy["order_reviews"].duplicated().sum()

np.int64(0)

In [34]:
copy["order_reviews"][
    copy["order_reviews"].duplicated(
        subset=["order_id"],
        keep=False
    )
].sort_values("order_id").head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11


## Check geolocation data

In [35]:
copy["geolocation"].isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [36]:
copy["geolocation"][[
    "geolocation_lat",
    "geolocation_lng"
]].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [37]:
# Check coordinates located outside the preliminary inspection zone around Brazil
geo_outside = copy["geolocation"][
    (copy["geolocation"]["geolocation_lat"] < -34)
    | (copy["geolocation"]["geolocation_lat"] > 6)
    | (copy["geolocation"]["geolocation_lng"] < -74)
    | (copy["geolocation"]["geolocation_lng"] > -34)
]

display(geo_outside)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [38]:
print("Number of rows:", len(copy["geolocation"]))

print("Number of unique ZIP codes:", copy["geolocation"]["geolocation_zip_code_prefix"].nunique())

Number of rows: 1000163
Number of unique ZIP codes: 19015


In [39]:
copy["geolocation"]["geolocation_zip_code_prefix"].value_counts().head(10)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
Name: count, dtype: int64[pyarrow]

## Create one location per ZIP code

In [40]:
# Filter Coordinates Within the Preliminary Bounds
geolocation_copy = copy["geolocation"].copy()

geo_valid = geolocation_copy[
    geolocation_copy["geolocation_lat"].between(-34, 6)
    & geolocation_copy["geolocation_lng"].between(-74, -34)
].copy()

In [41]:
# Calculate Median Coordinates per ZIP Code
geo_lookup = geo_valid.groupby(
    "geolocation_zip_code_prefix",
    as_index=False
)[["geolocation_lat", "geolocation_lng"]].median()

In [42]:
geo_lookup.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,01001,-23.550381,-46.634027
1,01002,-23.548551,-46.635072
2,01003,-23.548977,-46.635313
3,01004,-23.549535,-46.634771
4,01005,-23.549612,-46.636532


In [43]:
geo_lookup.shape

(19010, 3)

In [44]:
geo_lookup["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(0)

In [45]:
print(
    "ZIP codes without a representative location:",
    copy["geolocation"]["geolocation_zip_code_prefix"].nunique()
    - geo_lookup["geolocation_zip_code_prefix"].nunique()
)

ZIP codes without a representative location: 5


## Summarize Payments per Order

In [46]:
# Aggregate Payment Records
payment_summary = copy["order_payments"].groupby(
    "order_id",
    as_index=False
).agg(
    payment_total=("payment_value", "sum"),
    payment_record_count=("payment_sequential", "size"),
    payment_type_count=("payment_type", "nunique")
)

In [47]:
payment_summary.head()

,order_id,payment_total,payment_record_count,payment_type_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1


In [48]:
payment_summary["order_id"].duplicated().sum()

np.int64(0)

In [49]:
# Check payment amounts
original_total = copy["order_payments"]["payment_value"].sum()
summary_total = payment_summary["payment_total"].sum()

print("Original total:", original_total)
print("Summary total:", summary_total)

np.isclose(original_total, summary_total)

Original total: 16008872.120000001
Summary total: 16008872.12


np.True_

## Summarize Items per Order

In [50]:
item_summary = copy["order_items"].groupby(
    "order_id",
    as_index=False
).agg(
    total_qty=("order_item_id", "size"),
    product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"),
    item_value=("price", "sum"),
    shipping_fee=("freight_value", "sum")
)

In [51]:
# Calculate total order value
item_summary["total_order_value"] = (
    item_summary["item_value"]
    + item_summary["shipping_fee"]
).round(2)

item_summary.head()

,order_id,total_qty,product_count,seller_count,item_value,shipping_fee,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,218.04


In [52]:
item_summary["order_id"].duplicated().sum()

np.int64(0)

In [53]:
print(
    "Item count matches:",
    item_summary["total_qty"].sum() == len(copy["order_items"])
)

print(
    "Item total matches:",
    np.isclose(
        item_summary["item_value"].sum(),
        copy["order_items"]["price"].sum()
    )
)

print(
    "Freight total matches:",
    np.isclose(
        item_summary["shipping_fee"].sum(),
        copy["order_items"]["freight_value"].sum()
    )
)

Item count matches: True
Item total matches: True
Freight total matches: True


## Merge Orders with Customers

In [54]:
# Check Customer IDs
copy["customers"]["customer_id"].duplicated().sum()

np.int64(0)

In [55]:
# Merge the Tables
orders_merged = copy["orders"].merge(
    copy["customers"],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

In [56]:
# Check Matching Results
orders_merged["_merge"].value_counts()

_merge
both          99441
left_only         0
right_only        0
Name: count, dtype: int64

In [57]:
# Check Row Counts and Order IDs
print("Before merge:", len(copy["orders"]))
print("After merge:", len(orders_merged))

print(
    "Duplicate order IDs:",
    orders_merged["order_id"].duplicated().sum()
)

Before merge: 99441
After merge: 99441
Duplicate order IDs: 0


In [58]:
orders_merged = orders_merged.drop(columns="_merge")
orders_merged.head()

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,santo andre,SP


## Merge Order Summaries

### Merge the Item Summary

In [59]:
orders_merged = orders_merged.merge(
    item_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

### Merge the Payment Summary

In [60]:
orders_merged = orders_merged.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

**Money columns (BRL):** `item_value` is the sum of item prices per order; `shipping_fee` is the sum of item freight charges per order. `total_order_value = item_value + shipping_fee`, rounded to two decimal places. `payment_total` is aggregated separately from payment records. The two totals may differ or have different missing values; retain both for reconciliation and select only one when duplicate information is unnecessary for EDA/modeling.

### Check Row Counts and Order IDs

In [61]:
print("Original orders:", len(copy["orders"]))
print("After merging:", len(orders_merged))

print(
    "Duplicate order IDs:",
    orders_merged["order_id"].duplicated().sum()
)

Original orders: 99441
After merging: 99441
Duplicate order IDs: 0


### Check Orders Without Items or Payments

In [62]:
print(
    "Orders without item records:",
    orders_merged["total_qty"].isna().sum()
)

print(
    "Orders without payment records:",
    orders_merged["payment_record_count"].isna().sum()
)

Orders without item records: 775
Orders without payment records: 1


In [63]:
orders_merged[
    orders_merged["total_qty"].isna()
][["order_id", "order_status"]].head()

,order_id,order_status
266,8e24261a7e58791d10cb1bf9da94df5c,unavailable
586,c272bcd21c287498b4883c7512019702,unavailable
687,37553832a3a89c9b2db59701c357ca67,unavailable
737,d57e15fb07fd180f06ab3926b39edcd2,unavailable
1130,00b1cb0320190ca0daa2c88b35206009,canceled


In [64]:
orders_merged.head()

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_state,total_qty,product_count,seller_count,item_value,shipping_fee,total_order_value,payment_total,payment_record_count,payment_type_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,...,SP,1.0,1.0,1.0,29.99,8.72,38.71,38.71,3.0,2.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,BA,1.0,1.0,1.0,118.70,22.76,141.46,141.46,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,GO,1.0,1.0,1.0,159.90,19.22,179.12,179.12,1.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,RN,1.0,1.0,1.0,45.00,27.20,72.20,72.20,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,...,SP,1.0,1.0,1.0,19.90,8.72,28.62,28.62,1.0,1.0


## Merge Customer Locations

### Rename the Location Columns

In [65]:
customer_locations = geo_lookup.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
})

### Merge Customer Locations

In [66]:
orders_merged = orders_merged.merge(
    customer_locations,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

### Check Row Counts and Order IDs

In [67]:
print("Original orders:", len(copy["orders"]))
print("After merging:", len(orders_merged))

print(
    "Duplicate order IDs:",
    orders_merged["order_id"].duplicated().sum()
)

Original orders: 99441
After merging: 99441
Duplicate order IDs: 0


### Check Missing Customer Coordinates

In [68]:
orders_merged[[
    "customer_lat",
    "customer_lng"
]].isna().sum()

customer_lat    279
customer_lng    279
dtype: int64

In [69]:
orders_merged[
    orders_merged["customer_lat"].isna()
    | orders_merged["customer_lng"].isna()
][[
    "order_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]].head()

,order_id,customer_zip_code_prefix,customer_city,customer_state
445,690199d6a2c51ff57c6b392d7680cbfd,87323,alto sao joao,PR
610,9f2da077d2d1eb6e8abcacef8776e941,73081,brasilia,DF
681,1b00c4d80c81cdbc36487c96c6e92f07,28160,santo eduardo,RJ
728,8ad3f1d0f96992e43566c4c82c9f6c58,83252,ilha dos valadares,PR
926,7ebe19943d85e81a7b0eb108681276b7,71551,brasilia,DF


In [70]:
orders_merged.head(5)

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_count,seller_count,item_value,shipping_fee,total_order_value,payment_total,payment_record_count,payment_type_count,customer_lat,customer_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,...,1.0,1.0,29.99,8.72,38.71,38.71,3.0,2.0,-23.576170,-46.587276
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,118.70,22.76,141.46,141.46,1.0,1.0,-12.126651,-45.008162
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,159.90,19.22,179.12,179.12,1.0,1.0,-16.744472,-48.514624
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,45.00,27.20,72.20,72.20,1.0,1.0,-5.774611,-35.273916
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,09195,...,1.0,1.0,19.90,8.72,28.62,28.62,1.0,1.0,-23.675316,-46.515116


In [71]:
orders_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchased_at             99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_at     97658 non-null  datetime64[us]
 6   order_delivered_customer_at    96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_unique_id             99441 non-null  str           
 9   customer_zip_code_prefix       99441 non-null  string        
 10  customer_city                  99441 non-null  str           
 11  customer_state            

## Calculate Delivery Time and Delay

### Calculate Delivery Time

In [72]:
date_columns = [
    "order_purchased_at",
    "order_approved_at",
    "order_delivered_carrier_at",
    "order_delivered_customer_at",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_merged[col] = pd.to_datetime(
        orders_merged[col],
        errors="raise"
    )

In [73]:
orders_merged["delivery_time_days"] = (
    orders_merged["order_delivered_customer_at"]
    - orders_merged["order_purchased_at"]
).dt.total_seconds() / 86400

### Calculate Delivery Delay

In [74]:
orders_merged["delivery_delay_days"] = (
    orders_merged["order_delivered_customer_at"].dt.normalize()
    - orders_merged["order_estimated_delivery_date"].dt.normalize()
).dt.days

In [75]:
orders_merged[[
    "order_id",
    "order_status",
    "order_purchased_at",
    "order_delivered_customer_at",
    "order_estimated_delivery_date",
    "delivery_time_days",
    "delivery_delay_days"
]].head()

,order_id,order_status,order_purchased_at,order_delivered_customer_at,order_estimated_delivery_date,delivery_time_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.436574,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.782037,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.394213,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.208750,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.873877,-10.0


### Check Negative Delivery Times

In [76]:
orders_merged[orders_merged["delivery_time_days"] < 0]

,order_id,customer_id,order_status,order_purchased_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,item_value,shipping_fee,total_order_value,payment_total,payment_record_count,payment_type_count,customer_lat,customer_lng,delivery_time_days,delivery_delay_days


### Check Missing Delivery Times

In [77]:
orders_merged[orders_merged["delivery_time_days"].isna()]["order_status"].value_counts(dropna=False)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [78]:
# Check 8 delivered orders missing delivery time
orders_merged[
    (orders_merged["order_status"] == "delivered")
    & (orders_merged["delivery_time_days"].isna())
][[
    "order_id",
    "order_purchased_at",
    "order_delivered_customer_at"
]]

,order_id,order_purchased_at,order_delivered_customer_at
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,NaT
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,NaT
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,NaT


### Missing Delivery Dates

Eight orders marked as delivered have no recorded customer delivery date.
Their delivery times remain missing. These orders are retained for EDA.

## Merge Products with Category Translations

### Check Category Translation Keys

In [79]:
copy["category_translation"]["product_category_name"].duplicated().sum()

np.int64(0)

### Merge Category Translations

In [80]:
products_merged = copy["products"].merge(
    copy["category_translation"],
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

### Check Row Counts and Product IDs

In [81]:
print("Before merge:", len(copy["products"]))
print("After merge:", len(products_merged))

print(
    "Duplicate product IDs:",
    products_merged["product_id"].duplicated().sum()
)

Before merge: 32951
After merge: 32951
Duplicate product IDs: 0


### Check Categories Without Translations

In [82]:
products_merged[
    products_merged["product_category_name"].notna()
    & products_merged["product_category_name_english"].isna()
]["product_category_name"].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

### Create a Category Label for EDA

In [83]:
products_merged["product_category"] = (
    products_merged["product_category_name_english"]
    .fillna(products_merged["product_category_name"])
    .fillna("unknown")
)

## Merge Order Items with Products

### Merge Product Information

In [84]:
items_merged = copy["order_items"].merge(
    products_merged,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

### Check Matching Results

In [85]:
items_merged["_merge"].value_counts()

_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

In [86]:
items_merged[
    items_merged["_merge"] == "left_only"
][["order_id", "order_item_id", "product_id"]].head()

,order_id,order_item_id,product_id


### Check Row Counts and Item Keys

In [87]:
print("Before merge:", len(copy["order_items"]))
print("After merge:", len(items_merged))

print(
    "Duplicate item keys:",
    items_merged.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Before merge: 112650
After merge: 112650
Duplicate item keys: 0


### Remove the Merge Indicator

In [88]:
items_merged = items_merged.drop(columns="_merge")

items_merged.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,product_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,garden_tools


## Merge Order Items with Sellers

### Check Seller IDs

In [89]:
copy["sellers"]["seller_id"].duplicated().sum()

np.int64(0)

### Merge Seller Information

In [90]:
items_merged = items_merged.merge(
    copy["sellers"],
    on="seller_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

### Check Matching Results

In [91]:
items_merged["_merge"].value_counts()

_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

### Check Row Counts and Item Keys

In [92]:
print("Original items:", len(copy["order_items"]))
print("After merging:", len(items_merged))

print(
    "Duplicate item keys:",
    items_merged.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Original items: 112650
After merging: 112650
Duplicate item keys: 0


### Remove the Merge Indicator

In [93]:
items_merged = items_merged.drop(columns="_merge")
items_merged.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,product_category,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,pet_shop,03471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,garden_tools,87900,loanda,PR


## Merge Seller Locations

### Rename the Location Columns

In [94]:
seller_locations = geo_lookup.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})

### Merge Seller Locations

In [95]:
items_merged = items_merged.merge(
    seller_locations,
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

### Check Row Counts and Item Keys

In [96]:
print("Original items:", len(copy["order_items"]))
print("After merging:", len(items_merged))

print(
    "Duplicate item keys:",
    items_merged.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Original items: 112650
After merging: 112650
Duplicate item keys: 0


### Check Missing Seller Coordinates

In [97]:
items_merged[["seller_lat","seller_lng"]].isna().sum()

seller_lat    253
seller_lng    253
dtype: int64

In [98]:
items_merged[
    items_merged["seller_lat"].isna()
    | items_merged["seller_lng"].isna()
][[
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]].drop_duplicates().head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
159,2e90cb1677d35cfe24eef47d441b7c87,02285,sao paulo,SP
1255,870d0118f7a9d85960f29ad89d5d989a,37708,pocos de caldas,MG
2001,42bde9fef835393bb8a8849cb6b7f245,71551,brasilia,DF
4661,5962468f885ea01a1b6a97a218797b0a,82040,curitiba,PR
11663,0b3f27369a4d8df98f7eb91077e438ac,07412,aruja,SP


## Merge Items with Order Information

### Merge the Tables

In [99]:
items_merged = items_merged.merge(
    orders_merged,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

### Check Matching Results

In [100]:
items_merged["_merge"].value_counts()

_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64

### Check Row Counts and Item Keys

In [101]:
print("Original items:", len(copy["order_items"]))
print("After merging:", len(items_merged))

print(
    "Duplicate item keys:",
    items_merged.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Original items: 112650
After merging: 112650
Duplicate item keys: 0


### Remove the Merge Indicator

In [102]:
items_merged = items_merged.drop(columns="_merge")
items_merged.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,...,item_value,shipping_fee,total_order_value,payment_total,payment_record_count,payment_type_count,customer_lat,customer_lng,delivery_time_days,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,...,58.90,13.29,72.19,72.19,1.0,1.0,-21.762829,-41.310605,7.614421,-9.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,...,239.90,19.93,259.83,259.83,1.0,1.0,-20.205737,-50.926924,16.216181,-3.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,...,199.00,17.87,216.87,216.87,1.0,1.0,-19.870383,-44.594355,7.948437,-14.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,...,12.99,12.79,25.78,25.78,1.0,1.0,-23.104346,-46.595589,6.147269,-6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,...,199.90,18.14,218.04,218.04,1.0,1.0,-23.245512,-46.825182,25.114352,-16.0


## Create Variables for EDA

Keep reusable calculations in preparation. Orders remain one row per order, and items remain one row per order item. Missing values are retained, not replaced with invented measurements. These are EDA tables containing retrospective information; select predictors separately for modeling.

### Calculate Seller–Customer Distance in Miles

Use the Haversine formula with ZIP-code representative coordinates and an approximate Earth radius of 3,958.8 miles. Distances are great-circle approximations, not actual transportation routes. Formula reference: [scikit-learn Haversine distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.haversine_distances.html). A distance of zero can result from shared ZIP coordinates.

In [103]:
# Convert latitude and longitude from degrees to radians
seller_lat = np.radians(items_merged['seller_lat'])
seller_lng = np.radians(items_merged['seller_lng'])
customer_lat = np.radians(items_merged['customer_lat'])
customer_lng = np.radians(items_merged['customer_lng'])
lat_difference = customer_lat - seller_lat
lng_difference = customer_lng - seller_lng

a = (np.sin(lat_difference / 2) ** 2
     + np.cos(seller_lat) * np.cos(customer_lat)
     * np.sin(lng_difference / 2) ** 2)
items_merged['distance_miles'] = 3958.8 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))
items_merged['distance_missing'] = items_merged['distance_miles'].isna()
items_merged['same_state'] = (
    items_merged['seller_state'].astype('string')
    == items_merged['customer_state'].astype('string'))

# Each seller contributes once, even if they sell multiple items in an order.
seller_distances = items_merged[[
    'order_id', 'seller_id', 'distance_miles', 'same_state'
]].drop_duplicates(['order_id', 'seller_id'])
distance_summary = seller_distances.groupby('order_id', as_index=False).agg(
    mean_distance_miles=('distance_miles', 'mean'),
    max_distance_miles=('distance_miles', 'max'),
    sellers_with_distance=('distance_miles', 'count'),
    distance_seller_count=('seller_id', 'size'))
distance_summary['distance_complete'] = (
    distance_summary['sellers_with_distance'] == distance_summary['distance_seller_count'])
distance_summary['sellers_missing_distance'] = (
    distance_summary['distance_seller_count'] - distance_summary['sellers_with_distance'])
# Do not label a partial mean/max as the complete order distance.
distance_summary.loc[~distance_summary['distance_complete'],
    ['mean_distance_miles', 'max_distance_miles']] = np.nan
distance_summary = distance_summary.drop(columns='distance_seller_count')

### Calculate Product Weight and Volume Features

Source weights are converted from grams to pounds (1 lb = 453.59237 g) for the cleaned EDA tables. Dimensions are converted to inches (1 in = 2.54 cm); volume is in cubic inches (in³), where 1 in³ = 16.387064 cm³. `product_volume_in3` is the product of the three item dimensions, not actual parcel volume. Nonpositive or missing measurements are flagged and excluded from valid measurement summaries. Known sums describe only measured items. Complete totals are missing if any item has an invalid/missing measurement; no item is removed.

In [104]:
# Convert source grams to pounds: 1 lb = 453.59237 g.
# Keep full precision for aggregation; round only for display in EDA.
items_merged = items_merged.rename(columns={'product_weight_g': 'product_weight_lb'})
items_merged['product_weight_lb'] = items_merged['product_weight_lb'] / 453.59237

items_merged['weight_missing_or_nonpositive'] = (
    items_merged['product_weight_lb'].isna() | items_merged['product_weight_lb'].le(0))
# Convert source centimeters to inches: 1 inch = 2.54 cm.
items_merged = items_merged.rename(columns={
    'product_length_cm': 'product_length_in',
    'product_height_cm': 'product_height_in',
    'product_width_cm': 'product_width_in'
})
for col in ['product_length_in', 'product_height_in', 'product_width_in']:
    items_merged[col] = items_merged[col] / 2.54

dimension_columns = ['product_length_in', 'product_height_in', 'product_width_in']
items_merged['dimensions_missing_or_nonpositive'] = (
    items_merged[dimension_columns].isna().any(axis=1)
    | items_merged[dimension_columns].le(0).any(axis=1))
items_merged['product_volume_in3'] = (
    items_merged['product_length_in'] * items_merged['product_height_in']
    * items_merged['product_width_in']).where(
        ~items_merged['dimensions_missing_or_nonpositive'])
items_merged['category_missing'] = items_merged['product_category_name'].isna()

measurement_items = items_merged.copy()
measurement_items['valid_weight_lb'] = measurement_items['product_weight_lb'].where(
    ~measurement_items['weight_missing_or_nonpositive'])

def sum_known(values):
    return values.sum(min_count=1)

product_summary = measurement_items.groupby('order_id', as_index=False).agg(
    known_weight_lb=('valid_weight_lb', sum_known),
    mean_item_weight_lb=('valid_weight_lb', 'mean'),
    max_item_weight_lb=('valid_weight_lb', 'max'),
    items_missing_weight=('weight_missing_or_nonpositive', 'sum'),
    known_product_volume_in3=('product_volume_in3', sum_known),
    mean_item_volume_in3=('product_volume_in3', 'mean'),
    max_item_volume_in3=('product_volume_in3', 'max'),
    items_missing_volume=('dimensions_missing_or_nonpositive', 'sum'),
    items_missing_category=('category_missing', 'sum'),
    category_count=('product_category_name', 'nunique'),
)
product_summary['has_complete_weight'] = product_summary['items_missing_weight'].eq(0)
product_summary['has_complete_volume'] = product_summary['items_missing_volume'].eq(0)
product_summary['total_weight_lb'] = product_summary['known_weight_lb'].where(
    product_summary['has_complete_weight'])
product_summary['total_product_volume_in3'] = product_summary['known_product_volume_in3'].where(
    product_summary['has_complete_volume'])

orders_merged = (orders_merged
    .merge(distance_summary, on='order_id', how='left', validate='one_to_one')
    .merge(product_summary, on='order_id', how='left', validate='one_to_one'))
# No item records means the order has no complete distance or measurement summary.
for col in ['distance_complete', 'has_complete_weight', 'has_complete_volume']:
    orders_merged[col] = orders_merged[col].astype('boolean').fillna(False)

### Create Purchase-Time Variables and Delivery Flags

Purchase month includes year (YYYY-MM), while purchase_month is 1–12. Weekday uses Monday=0 through Sunday=6. is_late compares calendar delivery dates and remains missing if the comparison is unavailable. Delivery outcomes and quality flags are retrospective and must not be used as purchase-time predictors.

In [105]:
orders_merged['purchase_year'] = orders_merged['order_purchased_at'].dt.year.astype('Int64')
orders_merged['purchase_month'] = orders_merged['order_purchased_at'].dt.month.astype('Int64')
orders_merged['purchase_year_month'] = orders_merged['order_purchased_at'].dt.to_period('M').astype('string')
orders_merged['purchase_day_of_week'] = orders_merged['order_purchased_at'].dt.dayofweek.astype('Int64')
orders_merged['purchase_hour'] = orders_merged['order_purchased_at'].dt.hour.astype('Int64')
orders_merged['is_late'] = orders_merged['delivery_delay_days'].gt(0).astype('boolean').mask(
    orders_merged['delivery_delay_days'].isna())
orders_merged['delivery_target_valid'] = (
    orders_merged['order_status'].eq('delivered')
    & orders_merged['delivery_time_days'].notna()
    & orders_merged['delivery_time_days'].ge(0))
orders_merged['delivered_missing_actual'] = (
    orders_merged['order_status'].eq('delivered')
    & orders_merged['order_delivered_customer_at'].isna())
orders_merged['carrier_before_purchase'] = (
    orders_merged['order_delivered_carrier_at'] < orders_merged['order_purchased_at'])
orders_merged['carrier_before_approval'] = (
    orders_merged['order_delivered_carrier_at'] < orders_merged['order_approved_at'])
orders_merged['customer_before_carrier'] = (
    orders_merged['order_delivered_customer_at'] < orders_merged['order_delivered_carrier_at'])

# Copy only new order features to the item table, avoiding duplicate column names.
new_order_columns = [col for col in orders_merged.columns if col not in items_merged.columns]
items_merged = items_merged.merge(
    orders_merged[['order_id'] + new_order_columns],
    on='order_id', how='left', validate='many_to_one')
print('Orders:', orders_merged.shape)
print('Order items:', items_merged.shape)
print('Orders with complete distance:', int(orders_merged['distance_complete'].sum()))

Orders: (99441, 55)
Order items: (112650, 83)
Orders with complete distance: 98172


### Validate the New Features

In [106]:
assert len(orders_merged) == len(copy['orders'])
assert len(items_merged) == len(copy['order_items'])
assert orders_merged['order_id'].is_unique
assert not items_merged.duplicated(['order_id', 'order_item_id']).any()
assert np.isclose(items_merged['price'].sum(), copy['order_items']['price'].sum())
assert np.isclose(items_merged['freight_value'].sum(), copy['order_items']['freight_value'].sum())
assert orders_merged.loc[~orders_merged['distance_complete'], 'max_distance_miles'].isna().all()
assert orders_merged.loc[~orders_merged['has_complete_weight'], 'total_weight_lb'].isna().all()
assert orders_merged.loc[~orders_merged['has_complete_volume'], 'total_product_volume_in3'].isna().all()
assert orders_merged.loc[orders_merged['delivery_delay_days'].isna(), 'is_late'].isna().all()
print('New features verified; all source orders/items retained.')

New features verified; all source orders/items retained.


## Validate the Final Tables

### Check Row Counts

In [107]:
print("Original orders:", len(copy["orders"]))
print("Final orders:", len(orders_merged))

print("Original items:", len(copy["order_items"]))
print("Final items:", len(items_merged))

Original orders: 99441
Final orders: 99441
Original items: 112650
Final items: 112650


### Check Duplicate Keys

In [108]:
print(
    "Duplicate order IDs:",
    orders_merged["order_id"].duplicated().sum()
)

print(
    "Duplicate item keys:",
    items_merged.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Duplicate order IDs: 0
Duplicate item keys: 0


### Check Item Prices and Freight Totals

In [109]:
print(
    "Item prices preserved:",
    np.isclose(
        items_merged["price"].sum(),
        copy["order_items"]["price"].sum()
    )
)

print(
    "Freight values preserved:",
    np.isclose(
        items_merged["freight_value"].sum(),
        copy["order_items"]["freight_value"].sum()
    )
)

Item prices preserved: True
Freight values preserved: True


### Review Missing Values

In [110]:
print("Orders:")
display(orders_merged.isna().sum())

print("Order items:")
display(items_merged.isna().sum())

Orders:


order_id                            0
customer_id                         0
order_status                        0
order_purchased_at                  0
order_approved_at                 160
order_delivered_carrier_at       1783
order_delivered_customer_at      2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
total_qty                         775
product_count                     775
seller_count                      775
item_value                        775
shipping_fee                      775
total_order_value                 775
payment_total                       1
payment_record_count                1
payment_type_count                  1
customer_lat                      279
customer_lng                      279
delivery_time_days               2965
delivery_delay_days              2965
mean_distance_miles              1269
max_distance

Order items:


order_id                    0
order_item_id               0
product_id                  0
seller_id                   0
shipping_limit_date         0
                           ..
delivery_target_valid       0
delivered_missing_actual    0
carrier_before_purchase     0
carrier_before_approval     0
customer_before_carrier     0
Length: 83, dtype: int64

## Export the Cleaned Datasets

### Export Order–Product–Seller Lines
`clean_orders` retains order-level totals and customer location/time/delivery information. `clean_order_items` has one row per (order_id, product_id, seller_id), with qty and line-level amounts. Only customer_id and customer_unique_id are retained as customer attributes. Unit measurements and seller–customer distance remain available; total order attributes are not repeated. Price and product attributes must be constant within a group. If this assumption changes, the assertions stop execution for review.

In [111]:
# Build the final order-level table.
columns_to_remove = [
    'is_late', 'delivery_target_valid', 'delivered_missing_actual',
    'carrier_before_purchase', 'carrier_before_approval', 'customer_before_carrier',
    'sellers_with_distance', 'distance_complete', 'sellers_missing_distance',
    'items_missing_weight', 'items_missing_volume'
]
clean_orders = orders_merged.drop(columns=columns_to_remove).copy()
purchase_columns = [
    'purchase_year', 'purchase_month', 'purchase_year_month',
    'purchase_day_of_week', 'purchase_hour'
]

def arrange_order_columns(table):
    columns = [col for col in table.columns if col not in purchase_columns]
    position = columns.index('order_status') + 1
    columns[position:position] = purchase_columns
    columns.remove('customer_unique_id')
    columns.insert(columns.index('customer_id') + 1, 'customer_unique_id')
    columns.remove('customer_lat')
    columns.remove('customer_lng')
    position = columns.index('customer_state') + 1
    columns[position:position] = ['customer_lat', 'customer_lng']
    return table[columns]

clean_orders = arrange_order_columns(clean_orders)

# One row per product and seller within each order.
item_keys = ['order_id', 'product_id', 'seller_id']
grouped_items = items_merged.groupby(item_keys, sort=False)

# These fields must have a single value within each group.
# Stop instead of silently selecting one if future data contain conflicting values.
item_details = [
    'customer_id', 'customer_unique_id', 'shipping_limit_date',
    'product_category',
    'product_name_length', 'product_description_length', 'product_photos_qty',
    'product_weight_lb', 'product_length_in', 'product_height_in', 'product_width_in',
    'product_volume_in3', 'seller_zip_code_prefix', 'seller_city', 'seller_state',
    'seller_lat', 'seller_lng', 'distance_miles'
]
for col in ['price'] + item_details:
    assert grouped_items[col].nunique(dropna=False).le(1).all(), (
        f'Conflicting {col} values within order/product/seller group')

clean_order_items = grouped_items.agg(
    qty=('order_item_id', 'size'),
    unit_price=('price', 'first'),
    line_item_value=('price', 'sum'),
    line_shipping_fee=('freight_value', 'sum')
).reset_index()
product_details = grouped_items[item_details].first().reset_index()
clean_order_items = clean_order_items.merge(
    product_details, on=item_keys, how='left', validate='one_to_one')

# Source product weight/volume describe one unit; line totals include qty.
clean_order_items['line_weight_lb'] = (
    clean_order_items['product_weight_lb'] * clean_order_items['qty']).where(
        clean_order_items['product_weight_lb'].gt(0))
clean_order_items['line_product_volume_in3'] = (
    clean_order_items['product_volume_in3'] * clean_order_items['qty'])

front_columns = item_keys + ['qty', 'customer_id', 'customer_unique_id',
    'unit_price', 'line_item_value', 'line_shipping_fee']
other_columns = [col for col in clean_order_items.columns if col not in front_columns]
clean_order_items = clean_order_items[front_columns + other_columns]

# Validate grouped quantities and money against the original item rows.
assert not clean_order_items.duplicated(item_keys).any()
assert clean_order_items['qty'].sum() == len(copy['order_items'])
for old_col, new_col in [('price', 'line_item_value'), ('freight_value', 'line_shipping_fee')]:
    expected = copy['order_items'].groupby('order_id')[old_col].sum().sort_index()
    actual = clean_order_items.groupby('order_id')[new_col].sum().sort_index()
    pd.testing.assert_series_equal(expected, actual, check_names=False, check_dtype=False)
assert np.allclose(
    clean_order_items['unit_price'] * clean_order_items['qty'],
    clean_order_items['line_item_value'])

# Ensure order-level totals and customer location details are not repeated here.
assert not any(col in clean_order_items.columns for col in [
    'order_item_id', 'total_order_value', 'payment_total', 'item_value', 'shipping_fee',
    'total_qty', 'seller_count', 'total_weight_lb', 'total_product_volume_in3',
    'customer_zip_code_prefix', 'customer_city', 'customer_state', 'customer_lat', 'customer_lng'])
print('Order table:', clean_orders.shape)
print('Order-product-seller table:', clean_order_items.shape)
print('Total quantity:', int(clean_order_items['qty'].sum()))

clean_orders.to_csv(
    '/Users/anhtuyet15999/Projects/ecommerce_delivery_time/data/processed/clean_orders.csv',
    index=False)
clean_order_items.to_csv(
    '/Users/anhtuyet15999/Projects/ecommerce_delivery_time/data/processed/clean_order_items.csv',
    index=False)


Order table: (99441, 44)
Order-product-seller table: (102425, 27)
Total quantity: 112650
